# 六个涨跌停情绪因子：先择时、后 IC 分析

研究流程：构造因子与市场基准 → 历史阈值单因子择时 → IC 与滚动 IC。


In [ ]:
import sys
import warnings
from datetime import date
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "my_utils").is_dir():
        PROJECT_ROOT = candidate
        break
else:
    raise RuntimeError("未找到项目根目录：上级目录中缺少 my_utils/")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from my_utils.fun import read_day_data
from 因子回测.alpha import add_future_return
from 因子回测.涨跌停情绪因子.benchmark_loader import list_available_benchmarks
from 因子回测.涨跌停情绪因子.timing_engine import (
    analyze_ic,
    build_daily_sentiment_factors, build_value_weighted_benchmark,
    compute_rolling_ic, compute_threshold, plot_rolling_ic_history,
    plot_timing_nav_comparison, prepare_stock_daily, report_ic_summary,
    run_timing, summarize_timing, run_multi_benchmark_timing,
    plot_multi_benchmark_summary,
)

plt.rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "Arial Unicode MS"]
plt.rcParams["axes.unicode_minus"] = False
warnings.filterwarnings("ignore", message="Glyph .* missing from current font")

START_DATE = date(2018, 1, 2)
END_DATE = date(2026, 7, 20)
DATA_SOURCE = "rq_stock_all_data"
WINDOW = 5
HORIZONS = (1, 3, 5, 10)
ROLLING_IC_WINDOWS = (50, 100)
ROLLING_IC_MIN_VALID_RATIO = 0.95
IC_NEUTRAL_BAND = 0.02
THRESHOLD_QUANTILE = 0.60
MIN_HISTORY = 252
PRICE_TOLERANCE = 1e-6
FACTOR_LABELS = {
    "limit_up_ratio": "涨停占比", "limit_down_ratio": "跌停占比",
    "net_limit_ratio": "净涨停占比（涨跌停强度）",
    "limit_up_down_ratio": "涨跌停比值",
    "limit_up_next_ret": "涨停次日收益", "limit_down_next_ret": "跌停次日收益",
}
FACTOR_DIRECTIONS = {factor: 1 for factor in FACTOR_LABELS}
FACTOR_COLUMNS = list(FACTOR_LABELS)

# 研究参数保留在 Notebook，修改参数后可直接重跑。
# zz2000 当前数据源不可用；需要时可在此列表手动加入。
BENCHMARKS = ["all_a_value_weight", "zz500", "zz1000"]
BENCHMARK_LABELS = {
    "all_a_value_weight": "全A市值加权", "zz500": "中证500",
    "zz1000": "中证1000", "zz2000": "中证2000",
}


## 1. 构造六个情绪因子与市场基准


In [ ]:
daily_fields = [
    "code", "trading_date", "close", "pre_close", "limit_up", "limit_down",
    "is_st", "is_suspended", "total_mv",
]
daily_raw = read_day_data(
    START_DATE, END_DATE, fields=daily_fields, file_path=DATA_SOURCE,
).sort(["code", "trading_date"])

prepared_daily, trading_calendar = prepare_stock_daily(daily_raw)
factor_daily = build_daily_sentiment_factors(
    prepared_daily, trading_calendar, window=WINDOW,
)
market_daily = build_value_weighted_benchmark(prepared_daily, trading_calendar)

# 前瞻收益统一复用 alpha.add_future_return，保持其标准列名。
market_with_forward = add_future_return(
    market_daily, ret_col="market_daily_ret", horizons=HORIZONS,
)
research_data = (
    factor_daily.to_pandas()
    .assign(trading_date=lambda data: pd.to_datetime(data["trading_date"]))
    .merge(market_with_forward, on="trading_date", how="inner")
    .sort_values("trading_date")
    .reset_index(drop=True)
)
display(research_data[["trading_date", *FACTOR_COLUMNS]].tail())


## 2. 历史阈值与单因子择时

先验证阈值策略是否改善同期基准，再分析因子相关性。


In [ ]:
timing_input = compute_threshold(
    research_data, FACTOR_COLUMNS, THRESHOLD_QUANTILE,
    lower_quantile=0.65,
    upper_quantile=1,
    min_history=MIN_HISTORY,
)
for factor in FACTOR_COLUMNS:
    timing_input[f"signal_{factor}"] = (
        (timing_input[factor] >= timing_input[f"lower_{factor}"])
        & (timing_input[factor] <= timing_input[f"upper_{factor}"])
    ).astype(float)

threshold_columns = [f"lower_{factor}" for factor in FACTOR_COLUMNS]
common_valid = timing_input[threshold_columns].notna().all(axis=1)
common_anchor = pd.Timestamp(timing_input.loc[common_valid.idxmax(), "trading_date"])

timing_daily, timing_summary_rows = {}, []
for factor in FACTOR_COLUMNS:
    for horizon in HORIZONS:
        daily_detail, block_detail = run_timing(
            timing_input, signal_column=f"signal_{factor}", horizon=horizon,
            anchor_date=common_anchor,
        )
        timing_daily[(factor, horizon)] = daily_detail
        timing_summary_rows.append(summarize_timing(
            daily_detail, block_detail, factor=factor, horizon=horizon,
        ))

timing_summary = pd.DataFrame(timing_summary_rows)
timing_summary["annual_excess_return"] = (
    timing_summary["annual_return"] - timing_summary["benchmark_annual_return"]
)
display(timing_summary)
plot_timing_nav_comparison(timing_daily, start_date=common_anchor)

# 多基准回测：使用完全相同的因子与阈值规则，比较不同市场风格下的表现。
multi_results = run_multi_benchmark_timing(
    factor_daily=factor_daily, benchmarks=BENCHMARKS,
    prepared_daily=prepared_daily, calendar=trading_calendar,
    start_date=START_DATE, end_date=END_DATE, horizons=HORIZONS,
    lower_quantile=0.65, upper_quantile=1, min_history=MIN_HISTORY,
    factor_columns=FACTOR_COLUMNS, factor_labels=FACTOR_LABELS,
)
display(multi_results["summary"])
plot_multi_benchmark_summary(
    multi_results, factor_columns=FACTOR_COLUMNS, factor_labels=FACTOR_LABELS,
    horizons=HORIZONS, benchmarks=BENCHMARKS, benchmark_labels=BENCHMARK_LABELS,
)


## 3. 基础 IC 与滚动 IC


In [ ]:
ic_summary = analyze_ic(
    research_data, factor_columns=FACTOR_COLUMNS, horizons=HORIZONS,
    factor_directions=FACTOR_DIRECTIONS,
)
report_ic_summary(ic_summary)

rolling_ic_detail = compute_rolling_ic(
    research_data, factor_columns=FACTOR_COLUMNS, horizons=HORIZONS,
    windows=ROLLING_IC_WINDOWS, min_valid_ratio=ROLLING_IC_MIN_VALID_RATIO,
)
plot_rolling_ic_history(rolling_ic_detail, ic_summary)
